# 1. Environment Setup & Reproducibility
Install required packages, configure random seeds across all libraries for deterministic execution, and configure the compute device.

In [2]:
import os
import sys
import random
import copy
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm

# Add project root to path for src imports
repo_root = os.path.abspath(os.path.join(os.path.dirname('__file__'), '../..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src.utils import seed_everything, load_config
seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device configured: {device} | Seed: {seed_val}')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 61.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 62.4 MB/s eta 0:00:00
Configured device: cuda


# 2. Ontology Knowledge Graph Engine
Initializes the `OntologyEngine` to extract continuous 24-dimensional feature vectors representing emotions, cognitive appraisals, intensity, polarity, and negation cues.

In [3]:
# Initialize OntologyEngine using relative path to ontology RDF
from src.ontology_engine import OntologyEngine, VSFCOntologyEngine
from src.utils import load_config

config_path = os.path.join(repo_root, 'configs/vsfc_ekman.yaml')
config = load_config(config_path)
onto_cfg = config.get('ontology', {})

ontology_path = os.path.join(repo_root, onto_cfg.get('rdf_path', 'ontology/ekman_appraisal_ontology.rdf'))

ontengine = OntologyEngine(
    rdf_path=ontology_path,
    default_conf=onto_cfg.get('default_conf', 0.85),
    manual_boost=onto_cfg.get('manual_boost', 1.2),
    negation_attenuation=onto_cfg.get('negation_attenuation', 0.5),
    alpha_similarity=onto_cfg.get('alpha_similarity', 0.2),
    verbose=True,
)


Found ontology file at: /kaggle/input/datasets/minkduck/kg-ekman6/ekman6_4_8.rdf
Loading Knowledge Graph: /kaggle/input/datasets/minkduck/kg-ekman6/ekman6_4_8.rdf
Loaded 90420 triples.
Lexicon initialized with 2540 entries.


# 3. Lexical Coverage Analysis & Diagnostic Suite
Computes feature hit rates and active dimensions across datasets and verifies the engine against core linguistic diagnostic test cases.

In [4]:
# 1. Diagnostic test suite
test_cases = [
    ("tôi không hề thích nó", "Negation Scope Check"),
    ("cảm thấy không vui chút nào", "Flip Logic Check"),
    ("bún chả rất ngon", "Ambiguity Filter Check (Chả)"),
    ("làm hay nghỉ", "Ambiguity Filter Check (Hay)"),
    ("thất vọng tràn trề", "Multi-Word Expression Recognition"),
]

print("==================== ONTOLOGY DIAGNOSTIC REPORT ====================")
for text, desc in test_cases:
    vec, tokens, trace = ontengine.getvector(text, debug=True)
    matched = [t["phrase"] for t in trace] if trace else ["None"]
    is_neg = any(t["data"]["isnegation"] for t in trace) if trace else False

    labels = ontengine.ALLEMOTIONS
    emotions_vals = vec[:7]
    max_idx = np.argmax(emotions_vals)
    dominant = (
        f"{labels[max_idx]} ({emotions_vals[max_idx]:.4f})"
        if emotions_vals[max_idx] > 0
        else "Neutral/None"
    )

    print(f"\nScenario: {desc}")
    print(f"  - Input: '{text}'")
    print(f"  - Matched Phrases: {matched}")
    print(f"  - Negation Detected: {is_neg}")
    print(f"  - Dominant Emotion: {dominant}")
print("\n====================================================================")

==================== ONTOLOGY DIAGNOSTIC REPORT ====================

Scenario: Negation Scope Check
  - Input: 'tôi không hề thích nó'
  - Matched Phrases: ['không', 'thích']
  - Negation Detected: True
  - Dominant Emotion: Happiness (0.4510)

Scenario: Flip Logic Check
  - Input: 'cảm thấy không vui chút nào'
  - Matched Phrases: ['không', 'vui']
  - Negation Detected: True
  - Dominant Emotion: Happiness (0.4941)

Scenario: Ambiguity Filter Check (Chả)
  - Input: 'bún chả rất ngon'
  - Matched Phrases: ['rất']
  - Negation Detected: False
  - Dominant Emotion: Happiness (0.4189)

Scenario: Ambiguity Filter Check (Hay)
  - Input: 'làm hay nghỉ'
  - Matched Phrases: ['nghỉ']
  - Negation Detected: False
  - Dominant Emotion: Sadness (0.3888)

Scenario: Multi-Word Expression Recognition
  - Input: 'thất vọng tràn trề'
  - Matched Phrases: ['thất vọng']
  - Negation Detected: False
  - Dominant Emotion: Sadness (0.5764)



# 4. Dataset Loading, Preprocessing & DataLoaders
Loads the `vsfcekman-1` train, dev, and test sets, encodes the 7 Ekman emotion classes, extracts ontology features, and constructs PyTorch DataLoaders with balanced class weights.

In [5]:
tokenizer = AutoTokenizer.from_pretrained("uitnlp/visobert")


class EmotionDataset(Dataset):

    def __init__(self, df, tokenizer, ontology_engine, max_len=128):
        self.df = df
        self.tokenizer = tokenizer
        self.ont_engine = ontology_engine
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        raw_text = str(self.df.iloc[index]["Sentence"])
        label = self.df.iloc[index]["label_idx"]

        encoding = self.tokenizer(
            raw_text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt",
        )

        ont_vector = self.ont_engine.getvector(raw_text)
        if isinstance(ont_vector, tuple):
            ont_vector = ont_vector[0]

        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "ontology_features": torch.tensor(ont_vector, dtype=torch.float),
            "labels": torch.tensor(label, dtype=torch.long),
        }


# Dataset paths
BASE_DATA_PATH = "../../data/vsfc-ekman"

df_train = pd.read_csv(f"{BASE_DATA_PATH}/train.csv")
df_valid = pd.read_csv(f"{BASE_DATA_PATH}/dev.csv")
df_test = pd.read_csv(f"{BASE_DATA_PATH}/test.csv")

for df in [df_train, df_valid, df_test]:
    df.rename(
        columns={"Text": "Sentence", "Ekman_Label": "Emotion"}, inplace=True
    )
    df.dropna(subset=["Sentence", "Emotion"], inplace=True)

print(
    f"Loaded datasets: Train={len(df_train)}, Dev={len(df_valid)}, Test={len(df_test)}"
)

# Label Encoding
le = LabelEncoder()
all_labels = sorted(df_train["Emotion"].unique().astype(str))
le.fit(all_labels)
print(f"Classes ({len(le.classes_)}): {le.classes_}")

for df in [df_train, df_valid, df_test]:
    df["label_idx"] = le.transform(df["Emotion"].astype(str))

# DataLoaders
BATCH_SIZE = 32
train_loader = DataLoader(
    EmotionDataset(df_train, tokenizer, ontengine),
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,
)
valid_loader = DataLoader(
    EmotionDataset(df_valid, tokenizer, ontengine), batch_size=BATCH_SIZE
)
test_loader = DataLoader(
    EmotionDataset(df_test, tokenizer, ontengine), batch_size=BATCH_SIZE
)

# Compute balanced class weights
train_labels = df_train["label_idx"].tolist()
class_weights = compute_class_weight(
    "balanced", classes=np.unique(train_labels), y=train_labels
)
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f"Class Weights: {weights_tensor.cpu().numpy()}")

config.json:   0%|          | 0.00/644 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/471k [00:00<?, ?B/s]

HTTP Error 429 thrown while requesting HEAD https://huggingface.co/uitnlp/visobert/resolve/main/added_tokens.json
Rate limited. Waiting 8.0s before retry [Retry 1/5].


Loaded datasets: Train=11426, Dev=1583, Test=3166
Classes (7): ['Anger' 'Disgust' 'Fear' 'Happiness' 'Neutral' 'Sadness' 'Surprise']
Class Weights: [ 0.76168257  2.6327188  41.85348     0.28987494  3.571741    0.65448505
 38.863945  ]


# 5. Model Architectures
Defines neural-symbolic architectures for 7-class emotion classification:
1. `ViSoBERT_Fusion`: Standard ViSoBERT backbone supporting baseline, concatenation, and gated projection.
2. `ViSoBERT_Residual_Fusion`: Multi-layer ontology projector with gated residual connections and LayerNorm.
3. `OntologyConceptProjector` & `ViSoBERT_Ontology_CrossAttention` (OCA): Token-level multi-head cross-attention over 5 projected ontology concept tokens.

In [6]:
# Import proposed and benchmark model architectures from src.models
from src.models.final_comparison_models import (
    ViSoBERT_Baseline,
    ALDONAr_Model,
    KEAHT_Model,
    CombViSA_Model,
)

ONT_INPUT_DIM = 24
print('✅ Models loaded from src.models.final_comparison_models: ViSoBERT_Baseline, ALDONAr_Model, KEAHT_Model, CombViSA_Model')


# 6. Training Pipeline & Evaluation Utilities
Defines the training loop with validation early stopping and best checkpoint restoration.

In [7]:
def evaluate(model, loader):
    """Compute accuracy and macro F1 score on a DataLoader."""
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            ont = batch["ontology_features"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(ids, mask, ont)
            _, pr = torch.max(outputs, dim=1)
            preds.extend(pr.cpu().numpy())
            targets.extend(labels.cpu().numpy())

    return accuracy_score(targets, preds), f1_score(
        targets, preds, average="macro"
    )


def train_experiment(model, name, epochs=15, patience=3):
    """Execute model training with weighted CrossEntropyLoss and early stopping."""
    print(f"\nTraining Model: {name}")
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model = model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    criterion = nn.CrossEntropyLoss(weight=weights_tensor)

    best_f1 = 0.0
    patience_counter = 0
    best_weights = copy.deepcopy(
        model.module.state_dict()
        if isinstance(model, nn.DataParallel)
        else model.state_dict()
    )

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        for batch in train_loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            ont = batch["ontology_features"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            outputs = model(ids, mask, ont)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        val_acc, val_f1 = evaluate(model, valid_loader)
        avg_loss = total_loss / len(train_loader)
        print(
            f"Epoch {epoch + 1:02d}/{epochs:02d} | Loss: {avg_loss:.4f} | Val F1: {val_f1:.4f} | Val Acc: {val_acc:.4f}",
            end=" ",
        )

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_weights = copy.deepcopy(
                model.module.state_dict()
                if isinstance(model, nn.DataParallel)
                else model.state_dict()
            )
            patience_counter = 0
            print("(New Best)")
        else:
            patience_counter += 1
            print(f"(Patience: {patience_counter}/{patience})")

        if patience_counter >= patience:
            print(f"Early stopping triggered after {epoch + 1} epochs.")
            break

    if isinstance(model, nn.DataParallel):
        model.module.load_state_dict(best_weights)
    else:
        model.load_state_dict(best_weights)

    return model

# 7. Model Training Across Configurations


In [8]:
NUM_CLASSES = len(le.classes_)

# 1. Baseline ViSoBERT
m_baseline = ViSoBERT_Baseline(NUM_CLASSES)
m_baseline = train_experiment(
    m_baseline, "1. Baseline (ViSoBERT)", epochs=15, patience=3
)

# 2. ALDONAr
m_aldonar = ALDONAr_Model(NUM_CLASSES, ont_dim=ONT_INPUT_DIM)
m_aldonar = train_experiment(
    m_aldonar, "2. ALDONAr (Dual-Gated Fusion)", epochs=15, patience=3
)

# 3. KEAHT
m_keaht = KEAHT_Model(NUM_CLASSES, ont_dim=ONT_INPUT_DIM, n_heads=8)
m_keaht = train_experiment(
    m_keaht,
    "3. KEAHT (Knowledge-Enriched Attention)",
    epochs=15,
    patience=3,
)

# 4. CombViSA
m_combvisa = CombViSA_Model(NUM_CLASSES, ont_dim=ONT_INPUT_DIM)
m_combvisa = train_experiment(
    m_combvisa,
    "4. CombViSA (ViSoBERT + BiLSTM + Ontology)",
    epochs=15,
    patience=3,
)

pytorch_model.bin:   0%|          | 0.00/390M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: uitnlp/visobert
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Training Model: 1. Baseline (ViSoBERT)


model.safetensors:   0%|          | 0.00/390M [00:00<?, ?B/s]

...

XLMRobertaModel LOAD REPORT from: uitnlp/visobert
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Training Model: 3. KEAHT (Knowledge-Enriched Attention)
Epoch 01/15 | Loss: 1.5228 | Val F1: 0.4722 | Val Acc: 0.6955 (New Best)
Epoch 02/15 | Loss: 1.0312 | Val F1: 0.4941 | Val Acc: 0.7423 (New Best)
Epoch 03/15 | Loss: 0.7505 | Val F1: 0.5249 | Val Acc: 0.7568 (New Best)
Epoch 04/15 | Loss: 0.5474 | Val F1: 0.5401 | Val Acc: 0.7410 (New Best)
Epoch 05/15 | Loss: 0.4038 | Val F1: 0.5593 | Val Acc: 0.7738 (New Best)
Epoch 06/15 | Loss: 0.2985 | Val F1: 0.5540 | Val Acc: 0.7713 (Patience: 1/3)
Epoch 07/15 | Loss: 0.2253 | Val F1: 0.5753 | Val Acc: 0.7884 (New Best)
Epoch 08/15 | Loss: 0.1734 | Val F1: 0.5381 | Val Acc: 0.7587 (Patience: 1/3)
Epoch 09/15 | Loss: 0.1291 | Val F1: 0.5613 | Val Acc: 0.7858 (Patience: 2/3)
Epoch 10/15 | Loss: 0.0914 | Val F1: 0.5951 | Val Acc: 0.8054 (New Best)
Epoch 11/15 | Loss: 0.0846 | Val F1: 0.5719 | Val Acc: 0.7871 (Patience: 1/3)
Epoch 12/15 | Loss: 0.0764 | Val F1: 0.5425 | Val Acc: 0.7764 (Patience: 2/3)
Epoch 13/15 | Loss: 0.0764 | Val F1: 0.556

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: uitnlp/visobert
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Training Model: 4. CombViSA (ViSoBERT + BiLSTM + Ontology)
Epoch 01/15 | Loss: 1.6803 | Val F1: 0.4398 | Val Acc: 0.6481 (New Best)
Epoch 02/15 | Loss: 1.2787 | Val F1: 0.4990 | Val Acc: 0.7322 (New Best)
Epoch 03/15 | Loss: 1.0418 | Val F1: 0.5177 | Val Acc: 0.7473 (New Best)
Epoch 04/15 | Loss: 0.9015 | Val F1: 0.5446 | Val Acc: 0.7189 (New Best)
Epoch 05/15 | Loss: 0.6950 | Val F1: 0.5494 | Val Acc: 0.7631 (New Best)
Epoch 06/15 | Loss: 0.5342 | Val F1: 0.5619 | Val Acc: 0.7656 (New Best)
Epoch 07/15 | Loss: 0.4203 | Val F1: 0.5894 | Val Acc: 0.7846 (New Best)
Epoch 08/15 | Loss: 0.3394 | Val F1: 0.5673 | Val Acc: 0.7833 (Patience: 1/3)
Epoch 09/15 | Loss: 0.2610 | Val F1: 0.5440 | Val Acc: 0.7631 (Patience: 2/3)
Epoch 10/15 | Loss: 0.2192 | Val F1: 0.5720 | Val Acc: 0.7821 (Patience: 3/3)
Early stopping triggered after 10 epochs.


# 8. Final Benchmark Evaluation & Classification Reports
Collects model predictions on the test set and outputs macro F1, overall accuracy, and class-level classification metrics.

In [9]:
def get_predictions(model, loader):
    """Extract ground truth and model predictions on a dataset loader."""
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            ont = batch["ontology_features"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(ids, mask, ont)
            _, pr = torch.max(outputs, 1)
            preds.extend(pr.cpu().numpy())
            targets.extend(labels.cpu().numpy())
    return targets, preds


# Collect predictions across models
y_true, p_base = get_predictions(m_baseline, test_loader)
_, p_aldonar = get_predictions(m_aldonar, test_loader)
_, p_keaht = get_predictions(m_keaht, test_loader)
_, p_comb = get_predictions(m_combvisa, test_loader)

models_preds = [
    ("Baseline (ViSoBERT)", p_base),
    ("ALDONAr", p_aldonar),
    ("KEAHT", p_keaht),
    ("CombViSA", p_comb),
]

summary_records = []
for name, preds in models_preds:
    summary_records.append(
        {
            "Model Architecture": name,
            "Accuracy": accuracy_score(y_true, preds),
            "Macro F1": f1_score(y_true, preds, average="macro"),
        }
    )

print("\n==================== TEST SET BENCHMARK RESULTS ====================")
df_benchmark = pd.DataFrame(summary_records)
print(df_benchmark.round(4).to_string(index=False))

print("\n========== CLASSIFICATION REPORT: ALDONAr ==========")
print(
    classification_report(y_true, p_aldonar, target_names=le.classes_, digits=4)
)

print("\n========== CLASSIFICATION REPORT: KEAHT ==========")
print(
    classification_report(y_true, p_keaht, target_names=le.classes_, digits=4)
)

print("\n========== CLASSIFICATION REPORT: CombViSA ==========")
print(
    classification_report(y_true, p_comb, target_names=le.classes_, digits=4)
)


==================== TEST SET BENCHMARK RESULTS ====================
 Model Architecture  Accuracy  Macro F1
Baseline (ViSoBERT)    0.7660    0.5811
            ALDONAr    0.7486    0.5905
              KEAHT    0.7682    0.6092
           CombViSA    0.7675    0.5464

========== CLASSIFICATION REPORT: ALDONAr ==========
              precision    recall  f1-score   support

       Anger     0.6873    0.6181    0.6508       576
     Disgust     0.2616    0.5328    0.3510       137
        Fear     0.5455    0.4615    0.5000        13
   Happiness     0.9238    0.9243    0.9240      1586
     Neutral     0.4423    0.4132    0.4272       167
     Sadness     0.6507    0.5809    0.6138       680
    Surprise     0.6250    0.7143    0.6667         7

    accuracy                         0.7486      3166
   macro avg     0.5909    0.6064    0.5905      3166
weighted avg     0.7658    0.7486    0.7544      3166


========== CLASSIFICATION REPORT: KEAHT ==========
              precision    

# 9. Neuro-Symbolic Rule-Based Post-Processing
Evaluates an override reasoning mechanism applying high-confidence ontology rules over uncertain baseline model outputs.

In [10]:
class NeuroSymbolicReasoner:
    """Combines neural probability outputs with symbolic ontology rules."""

    def __init__(
        self,
        model,
        engine,
        tokenizer,
        device,
        le_classes,
        ont_threshold=0.6,
        model_conf_limit=0.9,
    ):
        self.model = (
            model.module if isinstance(model, nn.DataParallel) else model
        )
        self.engine = engine
        self.tokenizer = tokenizer
        self.device = device
        self.classes = le_classes
        self.cls_to_idx = {name: i for i, name in enumerate(le_classes)}
        self.ont_threshold = ont_threshold
        self.model_conf_limit = model_conf_limit

    def infer(self, text):
        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=128,
            padding="max_length",
        )
        input_ids = inputs["input_ids"].to(self.device)
        attention_mask = inputs["attention_mask"].to(self.device)

        vec = self.engine.getvector(text)
        if isinstance(vec, tuple):
            vec = vec[0]
        ont_tensor = torch.tensor(np.array([vec]), dtype=torch.float).to(
            self.device
        )

        self.model.eval()
        with torch.no_grad():
            outputs = self.model(input_ids, attention_mask, ont_tensor)
            probs = F.softmax(outputs, dim=1)

        pred_idx = torch.argmax(probs).item()
        confidence = probs[0][pred_idx].item()
        pred_label_str = self.classes[pred_idx]

        final_idx = pred_idx

        # Rule: If prediction is Neutral with moderate confidence, check for strong emotion hits
        if (
            pred_label_str == "Neutral"
            and confidence < self.model_conf_limit
        ):
            ont_emotions = vec[:7]
            max_ont_idx = np.argmax(ont_emotions)
            max_ont_val = ont_emotions[max_ont_idx]
            max_ont_label = self.engine.ALLEMOTIONS[max_ont_idx]

            if max_ont_val > self.ont_threshold and max_ont_label != "Neutral":
                if max_ont_label in self.cls_to_idx:
                    final_idx = self.cls_to_idx[max_ont_label]

        return pred_idx, final_idx


reasoner = NeuroSymbolicReasoner(
    model=m_baseline,
    engine=ontengine,
    tokenizer=tokenizer,
    device=device,
    le_classes=le.classes_,
    ont_threshold=0.6,
    model_conf_limit=0.9,
)

y_base = []
y_rule = []

for txt in tqdm(df_test["Sentence"], desc="Rule Post-processing"):
    p_b, p_r = reasoner.infer(txt)
    y_base.append(p_b)
    y_rule.append(p_r)

acc_base = accuracy_score(y_true, y_base)
f1_base = f1_score(y_true, y_base, average="macro")
acc_rule = accuracy_score(y_true, y_rule)
f1_rule = f1_score(y_true, y_rule, average="macro")

df_rule_comparison = pd.DataFrame(
    {
        "Method": ["Baseline (ViSoBERT)", "Baseline + Ontology Rules"],
        "Accuracy": [acc_base, acc_rule],
        "Macro F1": [f1_base, f1_rule],
        "Delta": [0.0, f1_rule - f1_base],
    }
)

print("\n============ NEURO-SYMBOLIC RULE-BASED POST-PROCESSING ============")
print(df_rule_comparison.round(4).to_string(index=False))

total_overrides = sum(1 for a, b in zip(y_base, y_rule) if a != b)
print(
    f"\nTotal samples overridden: {total_overrides}/{len(df_test)} ({total_overrides / len(df_test) * 100:.2f}%)"
)

Rule Post-processing:   0%|          | 0/3166 [00:00<?, ?it/s]


============ NEURO-SYMBOLIC RULE-BASED POST-PROCESSING ============
                   Method  Accuracy  Macro F1  Delta
      Baseline (ViSoBERT)    0.7660    0.5811 0.0000
Baseline + Ontology Rules    0.7669    0.5816 0.0005

Total samples overridden: 7/3166 (0.22%)


# 10. Statistical Significance Testing (Paired Permutation Test)
Calculates exact permutation p-values over 10,000 resamples to evaluate statistical significance ($p < 0.05$) between model prediction vectors.

In [11]:
from src.significance_test import paired_permutation_test, macro_f1


comparisons = [
    ("Baseline vs ALDONAr", p_base, p_aldonar),
    ("Baseline vs KEAHT", p_base, p_keaht),
    ("Baseline vs CombViSA", p_base, p_comb),
    ("ALDONAr vs KEAHT", p_aldonar, p_keaht),
    ("KEAHT vs CombViSA", p_keaht, p_comb),
]

print(
    "\n============== STATISTICAL SIGNIFICANCE TESTS (10,000 ITERATIONS) =============="
)

results = []
for name, preds_a, preds_b in comparisons:
    f1_a, f1_b, diff_f1, p_val_f1 = paired_permutation_test(
        y_true, preds_a, preds_b, metric_func=macro_f1
    )
    acc_a, acc_b, diff_acc, p_val_acc = paired_permutation_test(
        y_true, preds_a, preds_b, metric_func=accuracy_score
    )

    sig_f1 = "Significant (p < 0.05)" if p_val_f1 < 0.05 else "Not Significant"
    sig_acc = (
        "Significant (p < 0.05)" if p_val_acc < 0.05 else "Not Significant"
    )

    results.append(
        {
            "Comparison": name,
            "F1 (Model A)": round(f1_a, 4),
            "F1 (Model B)": round(f1_b, 4),
            "Diff (F1)": round(diff_f1, 4),
            "p-val (F1)": round(p_val_f1, 5),
            "Sig (F1)": sig_f1,
            "p-val (Acc)": round(p_val_acc, 5),
            "Sig (Acc)": sig_acc,
        }
    )

df_perm = pd.DataFrame(results)
print(df_perm.to_string(index=False))
print(
    "\n================================================================================"
)


============== STATISTICAL SIGNIFICANCE TESTS (10,000 ITERATIONS) ==============
          Comparison  F1 (Model A)  F1 (Model B)  Diff (F1)  p-val (F1)               Sig (F1)  p-val (Acc)              Sig (Acc)
 Baseline vs ALDONAr        0.5811        0.5905     0.0094     0.61734        Not Significant      0.00940 Significant (p < 0.05)
   Baseline vs KEAHT        0.5811        0.6092     0.0281     0.12889        Not Significant      0.77252        Not Significant
Baseline vs CombViSA        0.5811        0.5464     0.0347     0.02260 Significant (p < 0.05)      0.84602        Not Significant
    ALDONAr vs KEAHT        0.5905        0.6092     0.0186     0.42916        Not Significant      0.00280 Significant (p < 0.05)
   KEAHT vs CombViSA        0.6092        0.5464     0.0628     0.00160 Significant (p < 0.05)      0.95790        Not Significant

